# 09 - Agentic RAG

## Scenario: Self-Reflective Knowledge Retrieval

Standard RAG (Retrieval-Augmented Generation) is a simple pipeline: Embed Query -> Vector Search -> Stuff into Prompt -> Generate. 

If the user's query is poorly phrased, the Vector Search returns junk, and the LLM hallucinates based on the junk. 

**Agentic RAG** fixes this by giving the LLM the *control* over the search process. If the results are bad, the LLM can rewrite the query and search again!

In [1]:
import os
from openai import OpenAI
import chromadb

# 1. Attempt to use the real API
if os.environ.get("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
else:
    # 2. Fallback to our local mock for students without keys
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    import sys
    import os
    sys.path.append(os.path.abspath("../../.."))
    from awsome_agents.mock_openai import MockOpenAI
    client = MockOpenAI()

# 3. Optional: Local LLMs
# If you prefer to use a local model like Llama 3 instead of the mock:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# Setup dummy ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="northstar_policies")
collection.add(
    documents=["The EU SLA requires 99.99% uptime.", "US billing cycles end on the 15th."],
    ids=["doc1", "doc2"]
)


## 1. The Search Tool

We expose the RAG database as a *Tool*, rather than forcing it into the context window by default.

In [2]:
def search_knowledge_base(query: str) -> str:
    print(f"  🔍 [Tool] Searching for: '{query}'")
    results = collection.query(query_texts=[query], n_results=1)
    if not results['documents'][0]:
        return "No results found."
    return results['documents'][0][0]


## 2. The Agentic Query Loop

Because it's an agent, if it searches for "UK uptime" and gets "US billing cycles", it can realize it made a mistake, self-reflect, and search again for "EU SLA".

In [3]:
# Mocking the agentic loop execution
print("User: What is the UK uptime requirement?\n")

print("Agent Thought: The UK is part of the EU region. I should search the knowledge base.")
result = search_knowledge_base("UK uptime")
print(f"Tool Output: {result}\n")

print("Agent Thought: That didn't return anything useful. I will rewrite my query to be broader.")
result2 = search_knowledge_base("EU SLA uptime")
print(f"Tool Output: {result2}\n")

print("Agent Final Answer: The SLA requires 99.99% uptime for the UK (EU region).")


User: What is the UK uptime requirement?

Agent Thought: The UK is part of the EU region. I should search the knowledge base.
  🔍 [Tool] Searching for: 'UK uptime'
Tool Output: The EU SLA requires 99.99% uptime.

Agent Thought: That didn't return anything useful. I will rewrite my query to be broader.
  🔍 [Tool] Searching for: 'EU SLA uptime'
Tool Output: The EU SLA requires 99.99% uptime.

Agent Final Answer: The SLA requires 99.99% uptime for the UK (EU region).


## Checkpoint

**1. What makes RAG "Agentic"?**
- A) Using a more expensive embedding model.
- B) Giving the LLM the ability to autonomously call the search tool, evaluate the results, and refine the query if necessary before answering.
- C) Adding more documents to the database.
- D) Using LangChain instead of LlamaIndex.
